# Customers table: cleaning

Customers table: cleaning code compiled from the notebook, in the order it was run.
Cells are separated by "# %%" so VS Code can run this file cell by cell,
or you can paste each block into its own notebook cell.

Run from the folder that contains raw/ and cleaned/ (your VS Code workspace root).

Decisions and findings for this table are written up in `cleaning_notes/README_customers.md`.

### 1. Imports and settings

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

### 2. Load the raw customers file

In [2]:
customers = pd.read_csv(RAW / 'olist_customers_dataset.csv')

### 3. First inspection: structure, nulls, preview

In [3]:
customers.info()
print(customers.isna().sum())      # expect all zeros
print(customers.head())

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b

### 4. Key analysis: customer_id is per order, customer_unique_id is per person

In [4]:
print(customers['customer_id'].is_unique)                  # expect True
print(customers['customer_id'].nunique())                  # 99,441
print(customers['customer_unique_id'].nunique())           # 96,096

orders_per_person = customers['customer_unique_id'].value_counts()
print(orders_per_person.head())                            # top buyer has 17 orders
print((orders_per_person > 1).sum())                       # 2,997 repeat customers
print((orders_per_person > 1).mean() * 100)                # about 3.12 (%)

True
99441
96096
customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
Name: count, dtype: int64
2997
3.1187562437562435


### 5. Join integrity with orders (both directions)

In [5]:
orders = pd.read_csv(CLEAN / 'orders_clean.csv', usecols=['customer_id'])
print(orders['customer_id'].isin(customers['customer_id']).value_counts())   # True 99,441
print(customers['customer_id'].isin(orders['customer_id']).all())            # True

customer_id
True    99441
Name: count, dtype: int64
True


### 6. Zip code prefixes: restore leading zeros (stored as text, 5 characters)

In [6]:
customers['customer_zip_code_prefix'] = (
    customers['customer_zip_code_prefix'].astype(str).str.zfill(5)
)
print(customers['customer_zip_code_prefix'].str.len().value_counts())        # only 5

customer_zip_code_prefix
5    99441
Name: count, dtype: int64


### 7. State and city checks

In [7]:
print(customers['customer_state'].nunique())               # 27
print((customers['customer_city'] != customers['customer_city'].str.strip()).sum())   # 0 stray spaces
print(customers.loc[customers['customer_city'].str.contains(r"[^a-z \-']", regex=True),
                    'customer_city'].unique())             # only 'quilometro 14 do mutum'

# City casing for display: 'sao paulo' -> 'Sao Paulo'
customers['customer_city'] = customers['customer_city'].str.title()
print(customers['customer_city'].nunique())                # still 4,119

27
0
<StringArray>
['quilometro 14 do mutum']
Length: 1, dtype: str
4119


### 8. Save the cleaned table and verify the file on disk

In [8]:
customers.to_csv(CLEAN / 'customers_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'customers_clean.csv', dtype={'customer_zip_code_prefix': str})
print(check.shape)                                         # expect (99441, 5)
print(check['customer_zip_code_prefix'].str.len().value_counts())   # only 5
print(check['customer_city'].nunique())                    # 4,119

(99441, 5)
customer_zip_code_prefix
5    99441
Name: count, dtype: int64
4119
